<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRYOLO_enson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.2/949.2 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
import os
import numpy as np
import cv2
from ultralytics import YOLO
import random
import torch.nn.functional as F

In [36]:
class SuperResolution(nn.Module):
    def __init__(self):
        super(SuperResolution, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(32, 3, kernel_size=5, padding=2)
        self.upscale = nn.Upsample(scale_factor=2, mode='bicubic', align_corners=False)  # 256x512 -> 512x1024

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        x = self.upscale(x)  # Çıkışı HR boyuta yükselt
        return x.clamp(0, 1)

* Basit bir SRCNN mimarisi.
* Giriş ve çıkış kanalları (3) RGB görüntüler için uygun. Padding değerleri, giriş boyutunu koruyor.
* Çıkışta herhangi bir aktivasyon fonksiyonu (sigmoid ya da clamp) yok, eğer giriş görüntüleri [0,1] aralığında normalize edilmişse, çıkışın da bu aralıkta olması gerekir. Eğitim sırasında MSELoss ile uyumsuz olabilir.
* Önerilen çözüm eklendi.

In [37]:
"""
class SRYOLO(nn.Module):
    def __init__(self, yolo_weights):
        super(SRYOLO, self).__init__()
        self.sr = SuperResolution()
        self.yolo = YOLO(yolo_weights)
        self.sr_loss_fn = nn.MSELoss()

    def forward(self, x):
        sr_output = self.sr(x)
        # YOLO’ya vermeden önce boyutu 32’nin katı yap
        h, w = sr_output.shape[-2], sr_output.shape[-1]
        new_h = (h // 32) * 32
        new_w = (w // 32) * 32
        if h != new_h or w != new_w:
            sr_output = F.interpolate(sr_output, size=(new_h, new_w), mode='bicubic', align_corners=False)
        with torch.no_grad():
            results = self.yolo.predict(sr_output, verbose=False)
        return sr_output, results

    def compute_sr_loss(self, sr_output, high_res_target):
        return self.sr_loss_fn(sr_output, high_res_target)

    def train(self, mode=True):
        self.training = mode
        self.sr.train(mode)
        return self

    def eval(self):
        self.training = False
        self.sr.eval()
        return self
"""
class SRYOLO(nn.Module):
    def __init__(self, yolo_weights):
        super(SRYOLO, self).__init__()
        self.sr = SuperResolution()
        self.yolo = YOLO(yolo_weights)
        self.sr_loss_fn = nn.MSELoss()

    def forward(self, x):
        sr_output = self.sr(x)  # [B, 3, 256, 512] -> [B, 3, 512, 1024]
        with torch.no_grad():
            yolo_input = F.interpolate(sr_output, size=(256, 512), mode='bicubic', align_corners=False)  # YOLO için 256x512
            results = self.yolo.predict(yolo_input, verbose=False)
        return sr_output, results

    def compute_sr_loss(self, sr_output, high_res_target):
        return self.sr_loss_fn(sr_output, high_res_target)

*  SRCNN ile YOLOv8 birleştirilmiş. "forward" metodu, SR çıktısını YOLO'ya besliyor.
* **Potansiyel Sorun 1:** self.yolo(sr_output) çağrısı, YOLOv8’in PyTorch modülünden (ultralytics.YOLO) geliyor. Ancak ultralytics.YOLO modeli, genellikle bir PyTorch nn.Module gibi çalışmaz; daha çok bir wrapper gibi davranır ve giriş olarak tensor değil, görüntü listesi veya numpy array bekleyebilir. Bu durumda:
  * sr_output bir PyTorch tensor’ü (torch.Tensor), ama YOLOv8’in predict metodu çağrılmalı (self.yolo.predict(sr_output) gibi), ve bu metodun çıkışı da farklı bir formatta (örneğin, Results nesnesi) olabilir.
  * Eğitim sırasında backward() çalışması için YOLOv8’in kayıp fonksiyonunun da entegre edilmesi gerekir, ama burada sadece SR kaybı var.
* **Potansiyel Sorun 2:** results bir tensor değil, YOLOv8’in Results nesnesi olabilir. Bu nesne, backward() ile uyumlu değil. Eğer YOLOv8’i eğitmek istemiyorsan (sadece SR’yi eğitmek istiyorsan), bu sorun değil; ama kodu buna göre düzenlemek lazım.
  * Öneri: YOLOv8’in predict metodunu açıkça çağır ve eğitimi sadece SR için yapıyorsan, YOLO’yu eval modunda tut:


* **Görüntü Ön İşleme Pipeline'ı**
* Bu fonksiyonları bir ön işleme script'inde bir kez çalıştırılıp sonuçları diskte saklayacak bir fonksiyon yazılır. Daha sonra veri seti bu kaydedilmiş dosyaları okuyacak.
* downsample_image: PIL veya tensor girişi kabul eder, LR ve HR görüntüleri döndürür.
* apply_dark_channel_prior ve apply_cache: Tensor girişi kabul eder ve işlenmiş görüntüyü döndürür.

In [48]:
import os
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import cv2
import random

def downsample_image(img, target_lr_size=(256, 512), target_hr_size=(512, 1024), output_dir=None, img_name=None):
    if isinstance(img, torch.Tensor):
        lr_img = F.interpolate(img.unsqueeze(0), size=target_lr_size, mode='bicubic', align_corners=False).squeeze(0)
        hr_img = F.interpolate(img.unsqueeze(0), size=target_hr_size, mode='bicubic', align_corners=False).squeeze(0)
        if output_dir and img_name:
            lr_img_np = (lr_img.permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            hr_img_np = (hr_img.permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
            Image.fromarray(lr_img_np).save(os.path.join(output_dir, 'low_res', f'{img_name}_lr.jpg'))
            Image.fromarray(hr_img_np).save(os.path.join(output_dir, 'high_res', f'{img_name}_hr.jpg'))
        return lr_img, hr_img

def apply_dark_channel_prior(img, output_dir=None, img_name=None):
    """Dark Channel Prior algoritması uygulama"""
    if isinstance(img, torch.Tensor):
        np_img = img.permute(1, 2, 0).cpu().numpy()
    else:
        np_img = np.array(img) / 255.0

    if np_img.shape[2] == 3:
        np_img = np_img[:, :, ::-1]  # RGB -> BGR
    np_img = np.clip(np_img, 0.0, 1.0)
    min_channel = np.min(np_img, axis=2)
    kernel_size = 15
    dark_channel = cv2.erode(min_channel, np.ones((kernel_size, kernel_size)))
    num_pixels = dark_channel.size
    num_brightest = int(0.001 * num_pixels)
    flat_dark = dark_channel.flatten()
    flat_img = np_img.reshape(-1, 3)
    indices = np.argsort(flat_dark)[-num_brightest:]
    atmospheric = np.mean(flat_img[indices], axis=0)
    omega = 0.95
    transmission = 1 - omega * dark_channel / np.max(atmospheric)
    transmission = cv2.GaussianBlur(transmission, (kernel_size, kernel_size), 0)
    transmission = np.clip(transmission, 0.1, 1.0)
    result = np.zeros_like(np_img)
    for i in range(3):
        result[:, :, i] = (np_img[:, :, i] - atmospheric[i]) / transmission + atmospheric[i]
    result = np.clip(result, 0.0, 1.0)
    if result.shape[2] == 3:
        result = result[:, :, ::-1].copy()  # BGR -> RGB ve negatif stride’ı düzelt

    if output_dir and img_name:
        result_img = (result * 255).astype(np.uint8)
        Image.fromarray(result_img).save(os.path.join(output_dir, 'low_res', f'{img_name}_dcp.jpg'))

    if isinstance(img, torch.Tensor):
        return torch.from_numpy(result.copy()).permute(2, 0, 1).float()  # copy() ile stride düzeltiliyor
    return result * 255

def apply_clahe(img, output_dir=None, img_name=None):
    """CLAHE uygulama"""
    if isinstance(img, torch.Tensor):
        np_img = (img.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    else:
        np_img = np.array(img)

    if np_img.shape[2] == 3:
        np_img = cv2.cvtColor(np_img, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(np_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    enhanced_lab = cv2.merge((cl, a, b))
    enhanced_bgr = cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)
    enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

    if output_dir and img_name:
        Image.fromarray(enhanced_rgb).save(os.path.join(output_dir, 'low_res', f'{img_name}_clahe.jpg'))

    if isinstance(img, torch.Tensor):
        return torch.from_numpy(enhanced_rgb.copy()).permute(2, 0, 1).float() / 255.0  # copy() ile stride düzeltiliyor
    return enhanced_rgb

* **Ön İşleme Script'i**
* Script, orijinal görüntüleri okuyacak, downsample_image, apply_dark_channel_prior ve apple_cache fonksiyonlarını uygulayacak ve sonuçları diskte saklayacak.

In [39]:
!rm -rf /content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset/*
!rm -rf /content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset/*

In [40]:
def preprocess_images(image_dir, output_dir, target_lr_size=(256, 512), target_hr_size=(512, 1024)):
    os.makedirs(os.path.join(output_dir, 'low_res'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'high_res'), exist_ok=True)
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    transform = transforms.ToTensor()
    for img_name in image_files:
        img_id = os.path.splitext(img_name)[0]  # Temiz img_id
        img_path = os.path.join(image_dir, img_name)
        original_img = Image.open(img_path).convert('RGB')
        original_tensor = transform(original_img)
        downsample_image(original_tensor, target_lr_size, target_hr_size, output_dir, img_id)
    print(f"Ön işleme tamamlandı. Görüntüler {output_dir} klasörüne kaydedildi.")

# Örnek kullanım
image_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/train'
output_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset'
preprocess_images(image_dir, output_dir)

Ön işleme tamamlandı. Görüntüler /content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset klasörüne kaydedildi.


* val klasörü için preprocess işlemi

In [41]:
def preprocess_images(image_dir, output_dir, scale_factor=0.5):
    """Görüntüleri önceden işleyip kaydet"""
    os.makedirs(os.path.join(output_dir, 'low_res'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'high_res'), exist_ok=True)

    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    transform = transforms.ToTensor()

    for img_name in image_files:
        img_id = os.path.splitext(img_name)[0]
        img_path = os.path.join(image_dir, img_name)
        original_img = Image.open(img_path).convert('RGB')
        original_tensor = transform(original_img)

        # Downsample
        lr_img, hr_img = downsample_image(original_tensor, scale_factor, output_dir, img_id)

        # Rastgele bir işlem seç (veya hepsini yap)
        p = random.random()
        if p < 0.3:
            lr_img = apply_dark_channel_prior(lr_img, output_dir, img_id)
        elif p < 0.6:
            lr_img = apply_clahe(lr_img, output_dir, img_id)
        # Not: Eğer her zaman sadece downsample istiyorsan, bu koşulları kaldır.

    print(f"Ön işleme tamamlandı. Görüntüler {output_dir} klasörüne kaydedildi.")

# Örnek kullanım
image_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/val'
output_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset'
preprocess_images(image_dir, output_dir)

TypeError: upsample_bicubic2d() received an invalid combination of arguments - got (Tensor, list, bool, NoneType), but expected one of:
 * (Tensor input, tuple of ints output_size, bool align_corners, tuple of floats scale_factors)
      didn't match because some of the arguments have invalid types: (Tensor, !list of [float, float]!, bool, !NoneType!)
 * (Tensor input, tuple of ints output_size, bool align_corners, float scales_h = None, float scales_w = None, *, Tensor out = None)


In [ ]:
def preprocess_images(image_dir, output_dir, target_lr_size=(256, 512), target_hr_size=(512, 1024)): # Added target_lr_size and target_hr_size as parameters
    """Görüntüleri önceden işleyip kaydet"""
    os.makedirs(os.path.join(output_dir, 'low_res'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'high_res'), exist_ok=True)

    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    transform = transforms.ToTensor()

    for img_name in image_files:
        img_id = os.path.splitext(img_name)[0]
        img_path = os.path.join(image_dir, img_name)
        original_img = Image.open(img_path).convert('RGB')
        original_tensor = transform(original_img)

        # Downsample - Corrected the arguments
        lr_img, hr_img = downsample_image(original_tensor, target_lr_size, target_hr_size, output_dir, img_id)

        # Rastgele bir işlem seç (veya hepsini yap)
        p = random.random()
        if p < 0.3:
            lr_img = apply_dark_channel_prior(lr_img, output_dir, img_id)
        elif p < 0.6:
            lr_img = apply_clahe(lr_img, output_dir, img_id)
        # Not: Eğer her zaman sadece downsample istiyorsan, bu koşulları kaldır.

    print(f"Ön işleme tamamlandı. Görüntüler {output_dir} klasörüne kaydedildi.")

# Örnek kullanım
image_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/val'
output_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset'
preprocess_images(image_dir, output_dir) # Now using target_lr_size and target_hr_size

In [42]:
def preprocess_images(image_dir, output_dir, target_lr_size=(256, 512), target_hr_size=(512, 1024)): # Added target_lr_size and target_hr_size as parameters
    """Görüntüleri önceden işleyip kaydet"""
    os.makedirs(os.path.join(output_dir, 'low_res'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'high_res'), exist_ok=True)

    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    transform = transforms.ToTensor()

    for img_name in image_files:
        img_id = os.path.splitext(img_name)[0]
        img_path = os.path.join(image_dir, img_name)
        original_img = Image.open(img_path).convert('RGB')
        original_tensor = transform(original_img)

        # Downsample - Corrected the arguments
        lr_img, hr_img = downsample_image(original_tensor, target_lr_size, target_hr_size, output_dir, img_id)

        # Rastgele bir işlem seç (veya hepsini yap)
        p = random.random()
        if p < 0.3:
            lr_img = apply_dark_channel_prior(lr_img, output_dir, img_id)
        elif p < 0.6:
            lr_img = apply_clahe(lr_img, output_dir, img_id)
        # Not: Eğer her zaman sadece downsample istiyorsan, bu koşulları kaldır.

    print(f"Ön işleme tamamlandı. Görüntüler {output_dir} klasörüne kaydedildi.")

# Örnek kullanım
image_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/val'
output_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset'
preprocess_images(image_dir, output_dir) # Now using target_lr_size and target_hr_size

Ön işleme tamamlandı. Görüntüler /content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset klasörüne kaydedildi.


* **Görüntü İşleme Fonksiyonları:**
* apply_dark_channel_prior, apply_clahe, downsample_image: Bu fonksiyonlar detaylı ve genelde iyi yazılmış. Ancak:
  * Boyut Uyumu: downsample_image içinde F.interpolate ile LR görüntü türetiliyor ve geri büyütülüyor. Bu, eğitimde giriş-çıkış boyutlarının tutarlılığını sağlıyor.
  * Potansiyel Sorun: Eğer low_res_img ve original_img boyutları tam uyuşmazsa, MSELoss hata verebilir. Bunu kontrol etmeliyiz.

In [43]:
class SRYOLODataset(Dataset):
    def __init__(self, low_res_dir, high_res_dir, labels_dir, transform=None):
        self.low_res_dir = low_res_dir
        self.high_res_dir = high_res_dir
        self.labels_dir = labels_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(low_res_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
    def __len__(self):
        return len(self.image_files)
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        # Tüm ekleri temizle
        img_id = os.path.splitext(img_name)[0].replace('_lr', '').replace('_dcp', '').replace('_clahe', '')
        lr_path = os.path.join(self.low_res_dir, img_name)
        hr_path = os.path.join(self.high_res_dir, f'{img_id}_hr.jpg')
        low_res_img = Image.open(lr_path).convert('RGB')
        original_img = Image.open(hr_path).convert('RGB')
        label_path = os.path.join(self.labels_dir, f"{img_id}.txt")
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                labels = [list(map(float, line.strip().split())) for line in f.readlines()]
                labels = torch.tensor(labels)
        else:
            labels = torch.zeros((0, 5))
        if self.transform:
            low_res_img = self.transform(low_res_img)
            original_img = self.transform(original_img)
        return low_res_img, original_img, labels

* Veri seti kodu, HR görüntüyü yükleyip LR haline getiriyor ve etiketleri okuyor.
* **Potansiyel Sorun:**
  * labels değişkeni okunuyor ama geri dönüşte kullanılmıyor (return low_res_img, original_img, label_path). Eğer YOLOv8’in kaybını hesaplamak istiyorsan, etiketlerin de modele beslenmesi gerekir.
  * low_res_img ve original_img tensor’ler, ama label_path bir string. Eğitim sırasında bu, YOLO kaybını hesaplamada sorun çıkarabilir.

* **Öneri:** Eğer YOLO’yu eğitmeyeceksen, label_path’i döndürmek yerine etiketleri işleyip tensor formatına çevirmelisin (örneğin, YOLOv8’in beklediği bounding box formatında).


In [44]:
def train_sryolo(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device):
    model.to(device)
    best_val_loss = float('inf')
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for low_res_imgs, original_imgs, labels in train_loader:
            low_res_imgs = low_res_imgs.to(device)
            original_imgs = original_imgs.to(device)
            sr_outputs, yolo_results = model(low_res_imgs)
            sr_loss = model.compute_sr_loss(sr_outputs, original_imgs)
            loss = sr_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            torch.cuda.empty_cache()
        train_loss /= len(train_loader)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for low_res_imgs, original_imgs, labels in val_loader:
                low_res_imgs = low_res_imgs.to(device)
                original_imgs = original_imgs.to(device)
                sr_outputs, _ = model(low_res_imgs)
                val_loss += model.compute_sr_loss(sr_outputs, original_imgs).item()
        val_loss /= len(val_loader)
        scheduler.step()
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_sryolo.pth')

* Potansiyel Sorun 1: yolo_results bir Results nesnesi olabilir ve gradient hesaplamasına uygun değil. Eğer YOLO’yu eğitmiyorsan, forward içinde torch.no_grad() kullanmalısın (yukarıda önerdim).
* Potansiyel Sorun 2: loss = sr_loss sadece SR kaybını içeriyor. YOLOv8’in kaybını eklemek istiyorsan, bunu manuel olarak entegre etmen gerekir (örneğin, YOLO’nun loss fonksiyonunu çağırarak).
* Boyut Kontrolü: sr_outputs ve original_imgs boyutları uyuşmazsa MSELoss hata verir. Bunun için bir print(sr_outputs.shape, original_imgs.shape) ekleyip kontrol edebiliriz.

In [45]:
"""
def test_sryolo(model, test_loader, device, output_dir='outputs'):
    model.to(device)
    model.eval()
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/low_res", exist_ok=True)
    os.makedirs(f"{output_dir}/sr_output", exist_ok=True)
    os.makedirs(f"{output_dir}/original", exist_ok=True)
    os.makedirs(f"{output_dir}/yolo_result", exist_ok=True)

    psnr_values, ssim_values, detection_results = [], [], []

    with torch.no_grad():
        for i, (low_res_imgs, original_imgs, labels) in enumerate(test_loader):
            low_res_imgs = low_res_imgs.to(device)
            original_imgs = original_imgs.to(device)
            sr_outputs, yolo_results = model(low_res_imgs)

            # Yolo sonuçlarının türünü kontrol et
            print(f"Batch {i} - Yolo Results Type: {type(yolo_results)}, Length: {len(yolo_results)}")

            for j in range(len(low_res_imgs)):
                lr_img = low_res_imgs[j].cpu().numpy().transpose(1, 2, 0)
                lr_img = (lr_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(lr_img).save(f'{output_dir}/low_res/lr_input_{i}_{j}.jpg')

                sr_img = sr_outputs[j].cpu().numpy().transpose(1, 2, 0)
                sr_img = (sr_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(sr_img).save(f'{output_dir}/sr_output/sr_output_{i}_{j}.jpg')

                orig_img = original_imgs[j].cpu().numpy().transpose(1, 2, 0)
                orig_img = (orig_img * 255).clip(0, 255).astype(np.uint8)
                Image.fromarray(orig_img).save(f'{output_dir}/original/original_{i}_{j}.jpg')

                if isinstance(yolo_results, list) and hasattr(yolo_results[j], 'plot'):
                    result_img = yolo_results[j].plot()
                    Image.fromarray(result_img).save(f'{output_dir}/yolo_result/yolo_result_{i}_{j}.jpg')

                psnr_value = calculate_psnr(sr_img, orig_img)
                ssim_value = calculate_ssim(sr_img, orig_img)
                psnr_values.append(psnr_value)
                ssim_values.append(ssim_value)

                if isinstance(yolo_results, list) and hasattr(yolo_results[j], 'boxes') and hasattr(yolo_results[j].boxes, 'data'):
                    detection_results.append(yolo_results[j].boxes.data.cpu().numpy())
                else:
                    detection_results.append(np.array([]))

                print(f'Test Görüntüsü {i}_{j} işlendi. PSNR: {psnr_value:.2f}, SSIM: {ssim_value:.4f}')

    if psnr_values:
        avg_psnr = sum(psnr_values) / len(psnr_values)
        avg_ssim = sum(ssim_values) / len(ssim_values)
        print(f"\nTest sonuçları: Ortalama PSNR: {avg_psnr:.2f} dB, Ortalama SSIM: {avg_ssim:.4f}")
        analyze_detection_results(detection_results, output_dir)
        return avg_psnr, avg_ssim, detection_results
    else:
        print("Hiç test verisi işlenmedi!")
        return 0, 0, []
"""
def test_sryolo(model, val_loader, device):
    model.to(device)
    model.eval()
    with torch.no_grad():
        for low_res_imgs, original_imgs, labels in val_loader:
            low_res_imgs = low_res_imgs.to(device)
            sr_outputs, yolo_results = model(low_res_imgs)
            print("Test: SR Çıkış Boyutu:", sr_outputs.shape)
            print("Test: YOLO Sonuçları:", yolo_results[0].boxes.xyxy)
            break

In [12]:
# PSNR hesaplama fonksiyonu
def calculate_psnr(img1, img2):
    """İki görüntü arasındaki PSNR (Peak Signal-to-Noise Ratio) değerini hesaplar"""
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))
    return psnr

# SSIM hesaplama fonksiyonu
def calculate_ssim(img1, img2):
    """İki görüntü arasındaki SSIM (Structural Similarity Index) değerini hesaplar"""
    # Gri tonlamalı dönüşüm
    if img1.shape[2] == 3:
        img1_gray = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        img2_gray = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    else:
        img1_gray = img1
        img2_gray = img2

    C1 = (0.01 * 255) ** 2
    C2 = (0.03 * 255) ** 2

    # Ortalama
    mu1 = cv2.GaussianBlur(img1_gray, (11, 11), 1.5)
    mu2 = cv2.GaussianBlur(img2_gray, (11, 11), 1.5)

    # Varyans ve kovaryans
    mu1_sq = mu1 ** 2
    mu2_sq = mu2 ** 2
    mu1_mu2 = mu1 * mu2

    sigma1_sq = cv2.GaussianBlur(img1_gray ** 2, (11, 11), 1.5) - mu1_sq
    sigma2_sq = cv2.GaussianBlur(img2_gray ** 2, (11, 11), 1.5) - mu2_sq
    sigma12 = cv2.GaussianBlur(img1_gray * img2_gray, (11, 11), 1.5) - mu1_mu2

    # SSIM formülü
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()

# Nesne tespiti sonuçlarını analiz etme fonksiyonu
def analyze_detection_results(detection_results, output_dir):
    """YOLOv8 nesne tespiti sonuçlarını analiz eder ve raporlar"""
    class_counts = {}
    confidence_scores = []
    total_detections = 0

    for detections in detection_results:
        if len(detections) > 0:
            # Her bir tespit için
            for detection in detections:
                class_id = int(detection[5])
                confidence = detection[4]

                # Sınıf sayısını güncelle
                if class_id not in class_counts:
                    class_counts[class_id] = 0
                class_counts[class_id] += 1

                # Güven skorunu ekle
                confidence_scores.append(confidence)
                total_detections += 1

    # Sonuçları yazdır
    with open(f"{output_dir}/detection_analysis.txt", "w") as f:
        f.write("Nesne Tespiti Analizi\n")
        f.write("=====================\n\n")
        f.write(f"Toplam tespit edilen nesne sayısı: {total_detections}\n\n")

        f.write("Sınıf bazında tespit sayıları:\n")
        for class_id, count in class_counts.items():
            f.write(f"Sınıf {class_id}: {count} nesne\n")

        if confidence_scores:
            avg_confidence = sum(confidence_scores) / len(confidence_scores)
            f.write(f"\nOrtalama güven skoru: {avg_confidence:.4f}\n")
            f.write(f"Minimum güven skoru: {min(confidence_scores):.4f}\n")
            f.write(f"Maksimum güven skoru: {max(confidence_scores):.4f}\n")

    print(f"Tespit analizi '{output_dir}/detection_analysis.txt' dosyasına kaydedildi.")

In [46]:
def main():
  try:
    #YOLO ağırlık dosyası
    yolo_weights = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/training_logs/yolov8_high_alt/weights/best.pt'

    #Orijinal veri dizinleri
    data_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/train'
    labels_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/labels/train'
    val_data_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/images/val'
    val_labels_dir = '/content/drive/MyDrive/srcnn_dataset/altitude_dataset/high_alt_object_detection/labels/val'

    #Ön işlenmiş veri dizinleri
    preprocessed_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset'
    val_preprocessed_dir = '/content/drive/MyDrive/srcnn_dataset/SRYOLO/val_preprocessed_dataset'

    batch_size = 8
    num_epochs = 100
    learning_rate = 0.001
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"Cihaz: {device}")
    transform = transforms.ToTensor()
    def custom_collate_fn(batch):
        low_res_imgs = torch.stack([item[0] for item in batch])
        original_imgs = torch.stack([item[1] for item in batch])
        labels = [item[2] for item in batch]
        return low_res_imgs, original_imgs, labels

    # Eski dosyaları sil ve yeniden ön işleme yap
    """
    !rm -rf {preprocessed_dir}/*
    preprocess_images(data_dir, preprocessed_dir, target_size=(256, 512))
    """
    !rm -rf {preprocessed_dir}/*
    !rm -rf {preprocessed_val_dir}/*
    preprocess_images(data_dir, preprocessed_dir)
    preprocess_images(val_data_dir, preprocessed_val_dir)

    train_dataset = SRYOLODataset(
        low_res_dir=os.path.join(preprocessed_dir, 'low_res'),
        high_res_dir=os.path.join(preprocessed_dir, 'high_res'),
        labels_dir=labels_dir,
        transform=transform
    )
    val_dataset = SRYOLODataset(
        low_res_dir=os.path.join(preprocessed_val_dir, 'low_res'),
        high_res_dir=os.path.join(preprocessed_val_dir, 'high_res'),
        labels_dir=val_labels_dir,
        transform=transform
    )

    # Boyutları kontrol et
    for i in range(min(5, len(train_dataset))):
        lr, hr, lbl = train_dataset[i]
        print(f"LR shape: {lr.shape}, HR shape: {hr.shape}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=custom_collate_fn)

    print("Model oluşturuluyor...")
    model = SRYOLO(yolo_weights)
    print("Model başarıyla oluşturuldu!")
    print("SuperResolution Model Yapısı:", model.sr)

    params = [p for p in model.sr.parameters() if p.requires_grad]
    optimizer = optim.Adam(params, lr=learning_rate)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    print("Eğitim başlıyor...")
    train_sryolo(model, train_loader, val_loader, optimizer, scheduler, num_epochs, device)
    print("Test başlıyor...")
    test_sryolo(model, val_loader, device)
    print("İşlem tamamlandı!")

  except Exception as e:
    print(f"Hata oluştu: {e}")
    import traceback
    traceback.print_exc()

In [47]:
if __name__ == "__main__":
    main()

Cihaz: cuda
Ön işleme tamamlandı. Görüntüler /content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset klasörüne kaydedildi.
Ön işleme tamamlandı. Görüntüler /content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_val klasörüne kaydedildi.
LR shape: torch.Size([3, 256, 512]), HR shape: torch.Size([3, 512, 1024])
Hata oluştu: [Errno 2] No such file or directory: '/content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset/high_res/M0501_img000002_dcp_hr.jpg'


Traceback (most recent call last):
  File "<ipython-input-46-3419685c20b3>", line 60, in main
    lr, hr, lbl = train_dataset[i]
                  ~~~~~~~~~~~~~^^^
  File "<ipython-input-43-92f57fecbfff>", line 53, in __getitem__
    original_img = Image.open(hr_path).convert('RGB')
                   ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/PIL/Image.py", line 3465, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/srcnn_dataset/SRYOLO/preprocessed_dataset/high_res/M0501_img000002_dcp_hr.jpg'
